In [ ]:
%%writefile mnist_bags.py


"""
mnist_bags.py
MIL 학습 파이프라인 검증을 위한 합성 데이터셋 (MNIST-bags)
참고: Ilse et al. (2018), "Attention-based Deep Multiple Instance Learning"
"""

import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import Dataset


class MnistBags(Dataset):
    """
    각 bag(가방)은 여러 장의 MNIST 숫자 이미지(instance)로 구성됨.
    bag 안에 target_number(기본값 9)가 하나라도 있으면 bag label = 1, 없으면 0.
    -> 실제 병리 슬라이드(bag) 안에 암세포 패치(instance)가 있는지 여부를
       가장 단순한 형태로 흉내낸 합성 데이터.

    instance_labels_list: 각 instance의 실제 숫자 레이블 (디버깅/attention 검증용).
    실전 MIL(예: 병리 슬라이드)에서는 instance 단위 레이블에 접근할 수 없는 게 보통이지만,
    합성 데이터에서는 "attention이 정말 target에 집중했는지" 검증하는 데 쓸 수 있음.
    """

    def __init__(self, target_number=9, mean_bag_length=10, var_bag_length=2,
                 num_bags=250, seed=1, train=True):
        self.target_number = target_number
        self.mean_bag_length = mean_bag_length
        self.var_bag_length = var_bag_length
        self.num_bags = num_bags
        self.train = train

        self.r = np.random.RandomState(seed)

        loader = torch.utils.data.DataLoader(
            datasets.MNIST(
                './data', train=self.train, download=True,
                transform=transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.1307,), (0.3081,))
                ])
            ),
            batch_size=60000 if train else 10000, shuffle=False
        )
        self.all_imgs, self.all_labels = next(iter(loader))

        self.bags_list, self.labels_list, self.instance_labels_list = self._form_bags()

    def _form_bags(self):
        bags_list = []
        labels_list = []
        instance_labels_list = []

        for _ in range(self.num_bags):
            bag_length = int(self.r.normal(self.mean_bag_length, self.var_bag_length))
            bag_length = max(bag_length, 1)

            indices = torch.LongTensor(
                self.r.randint(0, len(self.all_imgs), bag_length)
            )
            bag_imgs = self.all_imgs[indices]
            bag_digit_labels = self.all_labels[indices]

            # bag label: target 숫자가 하나라도 포함되어 있으면 1
            bag_label = int((bag_digit_labels == self.target_number).any())

            bags_list.append(bag_imgs)
            labels_list.append(bag_label)
            instance_labels_list.append(bag_digit_labels)

        return bags_list, labels_list, instance_labels_list

    def __len__(self):
        return len(self.labels_list)

    def __getitem__(self, idx):
        bag = self.bags_list[idx]          # shape: (bag_length, 1, 28, 28)
        label = self.labels_list[idx]      # 0 또는 1
        return bag, label


if __name__ == "__main__":
    train_ds = MnistBags(num_bags=250, train=True, seed=1)
    test_ds = MnistBags(num_bags=50, train=False, seed=2)

    print(f"train bags: {len(train_ds)}, test bags: {len(test_ds)}")
    bag, label = train_ds[0]
    print(f"첫 번째 bag shape: {bag.shape}, label: {label}")
    print(f"positive bag 비율 (train): {sum(train_ds.labels_list) / len(train_ds):.2f}")

Writing mnist_bags.py


bag shape: torch.Size([13, 1, 28, 28]) -> shape 이 이렇게 생긴 이유

In [ ]:
!python mnist_bags.py

train bags: 250, test bags: 50
첫 번째 bag shape: torch.Size([13, 1, 28, 28]), label: 0
positive bag 비율 (train): 0.64


In [ ]:

%%writefile mil_model.py

"""
mil_model.py
Attention-based Deep MIL (Ilse et al., 2018) 구현
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class AttentionMIL(nn.Module):
    def __init__(self, in_channels=1, feature_dim=500, attention_dim=128):
        super().__init__()
        self.feature_dim = feature_dim
        self.attention_dim = attention_dim

        # 1. Instance feature extractor (작은 CNN)
        #    bag 안의 이미지 한 장(instance)을 벡터로 변환하는 역할
        #    AdaptiveAvgPool2d로 patch 크기가 달라져도(27x27, 32x32 등) 항상 4x4로 고정
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.feature_fc = nn.Sequential(
            nn.Linear(50 * 4 * 4, self.feature_dim),
            nn.ReLU(),
        )

        # 2. Attention 메커니즘: instance 임베딩마다 "중요도 점수"를 계산
        #    a_k = softmax( w^T tanh(V h_k) )
        self.attention_V = nn.Linear(self.feature_dim, self.attention_dim)
        self.attention_w = nn.Linear(self.attention_dim, 1)

        # 3. Bag-level classifier
        self.classifier = nn.Linear(self.feature_dim, 1)

    def forward(self, bag):
        # bag: (num_instances, 1, 28, 28)  <- 가방 하나에 들어있는 이미지들
        H = self.feature_extractor(bag)                # (N, 50, 4, 4)
        H = H.view(H.size(0), -1)                       # (N, 50*4*4)
        H = self.feature_fc(H)                           # (N, feature_dim) <- instance 임베딩

        # attention score 계산
        A = torch.tanh(self.attention_V(H))               # (N, attention_dim)
        A = self.attention_w(A)                            # (N, 1)
        A = torch.transpose(A, 1, 0)                        # (1, N)
        A = F.softmax(A, dim=1)                              # 가중치 합 = 1

        # attention 가중평균 -> bag을 대표하는 벡터 하나로 집계
        Z = torch.mm(A, H)                                    # (1, feature_dim)

        # bag-level 예측
        Y_logit = self.classifier(Z)                           # (1, 1)
        Y_prob = torch.sigmoid(Y_logit)

        return Y_prob, Y_logit, A


if __name__ == "__main__":
    from mnist_bags import MnistBags

    ds = MnistBags(num_bags=5, train=True, seed=1)
    model = AttentionMIL()

    bag, label = ds[0]
    prob, logit, attn = model(bag)

    print(f"bag shape: {bag.shape}, label: {label}")
    print(f"predicted prob: {prob.item():.4f}")
    print(f"attention weights shape: {attn.shape}, sum: {attn.sum().item():.4f}")

Writing mil_model.py


In [ ]:
!python mil_model.py

bag shape: torch.Size([13, 1, 28, 28]), label: 0
predicted prob: 0.5273
attention weights shape: torch.Size([1, 13]), sum: 1.0000


In [ ]:
%%writefile train.py

"""
train.py
MNIST-bags에서 Attention MIL 모델 학습 (Ilse et al. 2018 sanity check)
"""

import torch
import torch.nn.functional as F
import torch.optim as optim

from mnist_bags import MnistBags
from mil_model import AttentionMIL


def train(model, train_ds, optimizer, num_epochs=20):
    model.train()
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        correct = 0

        for bag, label in train_ds:
            label = torch.tensor([float(label)])

            optimizer.zero_grad()
            prob, logit, _ = model(bag)

            loss = F.binary_cross_entropy(prob.view(-1), label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pred = (prob.view(-1) > 0.5).float()
            correct += (pred == label).sum().item()

        train_loss = total_loss / len(train_ds)
        train_acc = correct / len(train_ds)
        print(f"[Epoch {epoch:02d}] train loss: {train_loss:.4f}, train acc: {train_acc:.4f}")


def evaluate(model, test_ds):
    model.eval()
    correct = 0
    with torch.no_grad():
        for bag, label in test_ds:
            label = torch.tensor([float(label)])
            prob, _, _ = model(bag)
            pred = (prob.view(-1) > 0.5).float()
            correct += (pred == label).sum().item()

    acc = correct / len(test_ds)
    print(f"Test accuracy: {acc:.4f} (error: {1 - acc:.4f})")
    return acc


if __name__ == "__main__":
    torch.manual_seed(1)

    train_ds = MnistBags(num_bags=250, train=True, seed=1)
    test_ds = MnistBags(num_bags=50, train=False, seed=2)

    model = AttentionMIL()
    optimizer = optim.Adam(
        model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=1e-4
    )

    train(model, train_ds, optimizer, num_epochs=20)
    evaluate(model, test_ds)

    torch.save(model.state_dict(), "attention_mil.pt")
    print("Saved trained model to attention_mil.pt")

Writing train.py


In [ ]:
!python train.py

[Epoch 01] train loss: 0.6594, train acc: 0.6200
[Epoch 02] train loss: 0.5958, train acc: 0.6560
[Epoch 03] train loss: 0.4023, train acc: 0.8200
[Epoch 04] train loss: 0.2789, train acc: 0.9040
[Epoch 05] train loss: 0.1941, train acc: 0.9240
[Epoch 06] train loss: 0.1328, train acc: 0.9520
[Epoch 07] train loss: 0.0797, train acc: 0.9760
[Epoch 08] train loss: 0.0443, train acc: 0.9960
[Epoch 09] train loss: 0.1038, train acc: 0.9600
[Epoch 10] train loss: 0.0638, train acc: 0.9880
[Epoch 11] train loss: 0.0367, train acc: 0.9960
[Epoch 12] train loss: 0.0251, train acc: 0.9960
[Epoch 13] train loss: 0.0201, train acc: 0.9960
[Epoch 14] train loss: 0.0158, train acc: 0.9960
[Epoch 15] train loss: 0.0123, train acc: 0.9960
[Epoch 16] train loss: 0.0096, train acc: 0.9960
[Epoch 17] train loss: 0.0029, train acc: 1.0000
[Epoch 18] train loss: 0.0022, train acc: 1.0000
[Epoch 19] train loss: 0.0015, train acc: 1.0000
[Epoch 20] train loss: 0.0012, train acc: 1.0000
Test accuracy: 0.940

In [ ]:
%%writefile visualize_attention.py

"""
visualize_attention.py
학습된 Attention MIL 모델이 실제로 target 숫자(9)에 높은 attention을
부여했는지 검증한다.
"""

import torch

from mnist_bags import MnistBags
from mil_model import AttentionMIL


def inspect_bag(model, bag, label, instance_labels, target_number=9):
    model.eval()
    with torch.no_grad():
        prob, _, attn = model(bag)

    attn = attn.view(-1)  # (N,)
    order = torch.argsort(attn, descending=True)

    print(f"bag label: {label}, predicted prob: {prob.item():.4f}")
    print(f"{'rank':<5}{'digit':<8}{'attention':<10}{'is target?'}")
    for rank, idx in enumerate(order):
        digit = instance_labels[idx].item()
        weight = attn[idx].item()
        is_target = "★" if digit == target_number else ""
        print(f"{rank:<5}{digit:<8}{weight:<10.4f}{is_target}")


if __name__ == "__main__":
    test_ds = MnistBags(num_bags=50, train=False, seed=2)

    model = AttentionMIL()
    model.load_state_dict(torch.load("attention_mil.pt", map_location="cpu"))

    # positive bag(라벨 1) 하나를 찾아서 검증
    for i in range(len(test_ds)):
        bag, label = test_ds[i]
        if label == 1:
            instance_labels = test_ds.instance_labels_list[i]
            print(f"=== test bag #{i} ===")
            inspect_bag(model, bag, label, instance_labels)
            break

Writing visualize_attention.py


In [ ]:
!python visualize_attention.py

=== test bag #1 ===
bag label: 1, predicted prob: 1.0000
rank digit   attention is target?
0    9       0.7418    ★
1    9       0.2577    ★
2    7       0.0003    
3    1       0.0001    
4    2       0.0000    
5    4       0.0000    
6    5       0.0000    
7    5       0.0000    
8    6       0.0000    


In [ ]:
!pip install gdown
!gdown --id 1RcNlwg0TwaZoaFO0uMXHFtAo_DCVPE6z

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1RcNlwg0TwaZoaFO0uMXHFtAo_DCVPE6z
From (redirected): https://drive.google.com/uc?id=1RcNlwg0TwaZoaFO0uMXHFtAo_DCVPE6z&confirm=t&uuid=26919327-9fdc-4ed7-85a5-daba4b64e870
To: /content/Patches.tar.gz
100% 41.2M/41.2M [00:00<00:00, 58.3MB/s]


In [ ]:
!tar -xzf Patches.tar.gz
!find . -maxdepth 3 -not -path '*/\.*' | head -50

.
./train.py
./Patches.tar.gz
./mil_model.py
./verify_attention_colon.py
./Patches
./Patches/0
./Patches/0/img84
./Patches/0/img44
./Patches/0/img92
./Patches/0/img73
./Patches/0/img64
./Patches/0/img29
./Patches/0/img89
./Patches/0/img99
./Patches/0/img11
./Patches/0/img45
./Patches/0/img98
./Patches/0/img39
./Patches/0/img59
./Patches/0/img28
./Patches/0/img94
./Patches/0/img37
./Patches/0/img90
./Patches/0/img26
./Patches/0/img62
./Patches/0/img95
./Patches/0/img83
./Patches/0/img93
./Patches/0/img82
./Patches/0/img12
./Patches/0/img100
./Patches/0/img96
./Patches/0/img24
./Patches/0/img97
./Patches/0/img74
./Patches/0/img23
./Patches/0/img2
./Patches/0/img16
./Patches/0/img25
./Patches/0/img35
./Patches/0/img81
./Patches/0/img58
./Patches/0/img3
./Patches/0/img34
./Patches/0/img63
./Patches/0/img4
./Patches/0/img33
./Patches/0/img1
./Patches/0/img56


In [ ]:
%%writefile colon_cancer_bags.py

"""
colon_cancer_bags.py
Colon Cancer MIL 데이터셋 로더
폴더 구조: Patches/<label>/<img_id>/<patch 파일들>
- label: 0(negative) 또는 1(positive)
- img_id 폴더 하나 = bag 하나
- 그 안의 개별 파일들 = instance(patch)
"""

import os
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class ColonCancerBags(Dataset):
    def __init__(self, root="Patches", transform=None):
        self.root = root
        self.transform = transform or transforms.Compose([
            transforms.ToTensor(),
        ])

        self.bag_paths = []   # 각 원소: 해당 bag(img 폴더) 안 patch 파일 경로 리스트
        self.labels = []

        for label_str in sorted(os.listdir(root)):        # "0", "1"
            label_dir = os.path.join(root, label_str)
            if not os.path.isdir(label_dir):
                continue
            label = int(label_str)

            for img_id in sorted(os.listdir(label_dir)):
                img_dir = os.path.join(label_dir, img_id)
                if not os.path.isdir(img_dir):
                    continue
                patch_files = [
                    os.path.join(img_dir, f)
                    for f in sorted(os.listdir(img_dir))
                    if os.path.isfile(os.path.join(img_dir, f))
                ]
                if len(patch_files) == 0:
                    continue

                self.bag_paths.append(patch_files)
                self.labels.append(label)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        patch_files = self.bag_paths[idx]
        label = self.labels[idx]

        instances = []
        for f in patch_files:
            img = Image.open(f).convert("RGB")
            instances.append(self.transform(img))

        bag = torch.stack(instances)   # (N, 3, H, W)
        return bag, label


if __name__ == "__main__":
    ds = ColonCancerBags(root="Patches")
    print(f"총 bag 수: {len(ds)}")
    print(f"positive bag 비율: {sum(ds.labels) / len(ds):.2f}")

    bag, label = ds[0]
    print(f"첫 bag shape: {bag.shape}, label: {label}")
    print(f"patch 파일 예시 경로: {ds.bag_paths[0][0]}")

Writing colon_cancer_bags.py


In [ ]:
!python colon_cancer_bags.py

총 bag 수: 99
positive bag 비율: 0.52
첫 bag shape: torch.Size([19, 3, 27, 27]), label: 0
patch 파일 예시 경로: Patches/0/img1/img1-xpos116-ypos95-fibroblast.bmp


In [ ]:
%%writefile train_colon.py

"""
train_colon.py
Colon Cancer 데이터셋에서 Attention MIL 모델 학습
"""

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Subset
from torchvision import transforms

from colon_cancer_bags import ColonCancerBags
from mil_model import AttentionMIL


def train(model, train_ds, optimizer, num_epochs=30):
    model.train()
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        correct = 0

        for bag, label in train_ds:
            label = torch.tensor([float(label)])

            optimizer.zero_grad()
            prob, logit, _ = model(bag)

            loss = F.binary_cross_entropy(prob.view(-1), label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pred = (prob.view(-1) > 0.5).float()
            correct += (pred == label).sum().item()

        train_loss = total_loss / len(train_ds)
        train_acc = correct / len(train_ds)
        print(f"[Epoch {epoch:02d}] train loss: {train_loss:.4f}, train acc: {train_acc:.4f}")


def evaluate(model, test_ds):
    model.eval()
    correct = 0
    with torch.no_grad():
        for bag, label in test_ds:
            label = torch.tensor([float(label)])
            prob, _, _ = model(bag)
            pred = (prob.view(-1) > 0.5).float()
            correct += (pred == label).sum().item()

    acc = correct / len(test_ds)
    print(f"Test accuracy: {acc:.4f} (error: {1 - acc:.4f})")
    return acc


if __name__ == "__main__":
    torch.manual_seed(1)

    # 학습용: flip/rotation/color jitter로 증강 -> "패치가 뒤집히거나 돌아가도
    # 세포 종류는 그대로"라는 사실을 모델이 배우게 함
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=180),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
    ])
    # 평가용: 증강 없이 원본 그대로 (공정한 평가를 위해)
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    # bag 개수와 순서만 확인하기 위한 메타데이터용 인스턴스
    meta_ds = ColonCancerBags(root="Patches")
    n_total = len(meta_ds)

    indices = torch.randperm(n_total, generator=torch.Generator().manual_seed(1)).tolist()
    n_train = int(n_total * 0.8)
    train_idx, test_idx = indices[:n_train], indices[n_train:]

    train_full = ColonCancerBags(root="Patches", transform=train_transform)
    test_full = ColonCancerBags(root="Patches", transform=eval_transform)

    train_ds = Subset(train_full, train_idx)
    test_ds = Subset(test_full, test_idx)
    print(f"train bags: {len(train_ds)}, test bags: {len(test_ds)}")

    model = AttentionMIL(in_channels=3)
    optimizer = optim.Adam(
        model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=1e-4
    )

    train(model, train_ds, optimizer, num_epochs=40)
    evaluate(model, test_ds)

    torch.save(model.state_dict(), "attention_mil_colon.pt")
    print("Saved trained model to attention_mil_colon.pt")


Writing train_colon.py


In [ ]:
!python train_colon.py

train bags: 79, test bags: 20
[Epoch 01] train loss: 0.6990, train acc: 0.5063
[Epoch 02] train loss: 0.6937, train acc: 0.4557
[Epoch 03] train loss: 0.6958, train acc: 0.5063
[Epoch 04] train loss: 0.6925, train acc: 0.5063
[Epoch 05] train loss: 0.6912, train acc: 0.5063
[Epoch 06] train loss: 0.6912, train acc: 0.5570
[Epoch 07] train loss: 0.6868, train acc: 0.5316
[Epoch 08] train loss: 0.6831, train acc: 0.5570
[Epoch 09] train loss: 0.6812, train acc: 0.5949
[Epoch 10] train loss: 0.6507, train acc: 0.6962
[Epoch 11] train loss: 0.6385, train acc: 0.6329
[Epoch 12] train loss: 0.6213, train acc: 0.6456
[Epoch 13] train loss: 0.5771, train acc: 0.6835
[Epoch 14] train loss: 0.5361, train acc: 0.7468
[Epoch 15] train loss: 0.4685, train acc: 0.8101
[Epoch 16] train loss: 0.4191, train acc: 0.7722
[Epoch 17] train loss: 0.4742, train acc: 0.7722
[Epoch 18] train loss: 0.4667, train acc: 0.7975
[Epoch 19] train loss: 0.4379, train acc: 0.7848
[Epoch 20] train loss: 0.4930, train ac

In [ ]:
%%writefile visualize_attention_colon.py

"""
visualize_attention_colon.py
학습된 Attention MIL 모델이 실제로 epithelial(상피) 세포 patch에
높은 attention을 부여했는지 검증한다.
파일명 형식: imgN-xposX-yposY-<celltype>.bmp 에서 celltype을 그대로 이용.
"""

import os
import torch

from colon_cancer_bags import ColonCancerBags
from mil_model import AttentionMIL


def cell_type_from_filename(path):
    # img1-xpos116-ypos95-fibroblast.bmp -> "fibroblast"
    name = os.path.basename(path)
    name = os.path.splitext(name)[0]
    return name.split("-")[-1]


def inspect_bag(model, bag, label, patch_paths):
    model.eval()
    with torch.no_grad():
        prob, _, attn = model(bag)

    attn = attn.view(-1)
    order = torch.argsort(attn, descending=True)

    print(f"bag label: {label}, predicted prob: {prob.item():.4f}")
    print(f"{'rank':<5}{'cell type':<14}{'attention':<10}{'epithelial?'}")
    for rank, idx in enumerate(order):
        cell_type = cell_type_from_filename(patch_paths[idx])
        weight = attn[idx].item()
        mark = "★" if cell_type == "epithelial" else ""
        print(f"{rank:<5}{cell_type:<14}{weight:<10.4f}{mark}")


if __name__ == "__main__":
    ds = ColonCancerBags(root="Patches")

    model = AttentionMIL(in_channels=3)
    model.load_state_dict(torch.load("attention_mil_colon.pt", map_location="cpu"))

    # positive bag(라벨 1) 하나를 찾아서 검증
    for i in range(len(ds)):
        bag, label = ds[i]
        if label == 1:
            patch_paths = ds.bag_paths[i]
            print(f"=== bag #{i} ({ds.bag_paths[i][0].split('/')[1]}) ===")
            inspect_bag(model, bag, label, patch_paths)
            break

Writing visualize_attention_colon.py


In [ ]:
!python visualize_attention_colon.py

=== bag #48 (1) ===
bag label: 1, predicted prob: 0.9040
rank cell type     attention epithelial?
0    epithelial    0.5294    ★
1    epithelial    0.2508    ★
2    epithelial    0.0413    ★
3    inflammatory  0.0294    
4    epithelial    0.0222    ★
5    fibroblast    0.0189    
6    epithelial    0.0166    ★
7    epithelial    0.0154    ★
8    epithelial    0.0104    ★
9    epithelial    0.0098    ★
10   epithelial    0.0084    ★
11   epithelial    0.0067    ★
12   epithelial    0.0065    ★
13   epithelial    0.0033    ★
14   epithelial    0.0029    ★
15   epithelial    0.0027    ★
16   epithelial    0.0024    ★
17   epithelial    0.0023    ★
18   epithelial    0.0023    ★
19   epithelial    0.0019    ★
20   epithelial    0.0019    ★
21   epithelial    0.0017    ★
22   epithelial    0.0012    ★
23   epithelial    0.0012    ★
24   epithelial    0.0009    ★
25   epithelial    0.0008    ★
26   epithelial    0.0007    ★
27   epithelial    0.0005    ★
28   epithelial    0.0005    ★
29   

In [ ]:
%%writefile verify_attention_colon.py
"""
verify_attention_colon.py
test에 속한 positive bag 전부에 대해, attention이 epithelial patch에
기준선(무작위로 뽑았을 때의 기대치)보다 더 몰리는지 확인한다.
"""

import os
import torch

from colon_cancer_bags import ColonCancerBags
from mil_model import AttentionMIL


def cell_type_from_filename(path):
    # img1-xpos116-ypos95-fibroblast.bmp -> "fibroblast"
    name = os.path.splitext(os.path.basename(path))[0]
    return name.split("-")[-1]


if __name__ == "__main__":
    ds = ColonCancerBags(root="Patches")
    n_total = len(ds)

    # [1] train_colon.py와 똑같은 셔플(시드 1)로 test에 속한 bag 번호를 복원
    indices = torch.randperm(n_total, generator=torch.Generator().manual_seed(1)).tolist()
    n_train = int(n_total * 0.8)
    test_idx = indices[n_train:]

    model = AttentionMIL(in_channels=3)
    model.load_state_dict(torch.load("attention_mil_colon.pt", map_location="cpu"))
    model.eval()

    print(f"{'bag':<6}{'n_patch':<9}{'baseline':<10}{'model_share':<13}{'diff'}")

    diffs = []
    for i in test_idx:
        bag, label = ds[i]
        if label != 1:            # [2] positive bag만 사용
            continue

        patch_paths = ds.bag_paths[i]
        is_epi = torch.tensor(
            [cell_type_from_filename(p) == "epithelial" for p in patch_paths]
        )

        with torch.no_grad():
            _, _, attn = model(bag)
        attn = attn.view(-1)

        baseline = is_epi.float().mean().item()   # [3] 기준선: epithelial 비율
        share = attn[is_epi].sum().item()          # [4] 모델의 몫: epithelial의 attention 합
        diff = share - baseline
        diffs.append(diff)

        print(f"{i:<6}{len(patch_paths):<9}{baseline:<10.2f}{share:<13.2f}{diff:+.2f}")

    # [5] 요약 숫자 두 개
    n_higher = sum(d > 0 for d in diffs)
    print(f"\n모델의 몫이 기준선보다 높은 bag: {len(diffs)}개 중 {n_higher}개")
    print(f"평균 차이: {sum(diffs) / len(diffs):+.3f}")

Overwriting verify_attention_colon.py


In [ ]:
!python verify_attention_colon.py

bag   n_patch  baseline  model_share  diff
80    326      0.73      0.96         +0.23
92    332      0.85      0.96         +0.11
50    207      0.68      1.00         +0.32
57    110      0.80      1.00         +0.20
61    84       0.30      0.99         +0.70
95    359      0.38      1.00         +0.62
69    187      0.37      1.00         +0.63
91    692      0.14      0.74         +0.60
93    505      0.50      0.98         +0.48

모델의 몫이 기준선보다 높은 bag: 9개 중 9개
평균 차이: +0.431
